In [ ]:
import numpy as np
import torch
import os 
import csv
import time
from functools import wraps
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM



Loads the model and tokenizer with optional quantization config so we can load either the traditional quantized model or the turbo quantized version.

In [ ]:
def load_model(model_id , quantization_config = None):
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  if tokenizer.pad_token is None : 
    tokenizer.pad_token = tokenizer.eos_token

  model = AutoModelForCausalLM.from_pretrained(
        model_id ,
        quantization_config = quantization_config ,
        device_map='auto',
        dtype = torch.float16
    )
  model.eval()
  return model , tokenizer

Measure GPU memory usage in GB per experiment

In [ ]:
def get_gpu_memory_usage():
  
  torch.cuda.synchronize()

  allocated = torch.cuda.memory_allocated() / (1024**3)
  reserved = torch.cuda.memory_reserved() / (1024**3)
  peak = torch.cuda.max_memory_allocated() / (1024**3)
  
  return allocated , reserved , peak

Computes the actual size of the KV cache in MB after a single forward pass 

In [ ]:
def get_kv_cache_size(model , tokenizer , prompt , device = 'cuda'):
    input_ids = tokenizer(prompt , return_tensors = 'pt').to(model.device)

    with torch.no_grad():
        out = model(**input_ids , use_cache = True)

    past = out.past_key_values

    total_bytes = 0

    for layer_kv in past : 
        k , v = layer_kv[0] , layer_kv[1]
        total_bytes += k.element_size() * k.nelement()
        total_bytes += v.element_size() * v.nelement()

    return {
        'number of layers ' : len(past),
        'kv_cache_ bytes' : total_bytes,
        'seq_len' : input_ids['input_ids'].shape[1]
    }

Measure generation speed; time to first token generation and time to full tokens generation 

In [ ]:
def measure_throughput(model , tokenizer , prompt , max_new_tokens= 100):
    input_ids = tokenizer(prompt , return_tensors = 'pt')

    with torch.no_grad():
        model.generate(**input_ids , max_new_tokens=1)

    
    torch.cuda.synchronize()


    start_timer = time.time()

    with torch.no_grad():
        model.generate(**input_ids , max_new_tokens = 1)
    
    torch.cuda.synchronize()
    time_to_first_token = time.time() - start_timer

    start_timer = time.time()
    
    with torch.no_grad():
        output = model.generate(**input_ids , max_new_tokens = max_new_tokens)

    torch.cuda.synchronize()

    time_to_full_generation = time.time() - start_timer

    generated_tokens = output.shape[1] - input_ids['input_ids'].shape[1]
    
    decode_time = time_to_full_generation - time_to_first_token
    decode_tokens_per_sec = (generated_tokens -1) / decode_time if decode_time > 0 else 0

    return {
        'time to first token' : time_to_first_token , 
        'total time to all tokens' : time_to_full_generation,
        'generated_tokens' : generated_tokens,
        'decode tokens per second' : decode_tokens_per_sec
    }


computes perplexity over a set of evaluation texts , to check how much quantization effects model quality 

In [ ]:
def measure_quality(model , tokenizer , texts):
    losses = []

    for text in texts:
        enc = tokenizer(text , return_tensors = 'pt')
        input_ids = enc['input_ids']

        with torch.no_grad():
            out = model(input_ids , labels = input_ids)

        losses.append(out.loss.item())

    avg_loss = np.mean(losses)
    perplexity = np.exp(avg_loss)

    return {
        'avg_loss' : avg_loss , 
        'perplexity' : perplexity
    }
        

Takes results and save them into a CSV file

In [ ]:
def save_results_to_csv(results , file_name):
    
    file_exists = os.path.isfile(file_name)

    with open(file_name , mode='a' , newline='') as csv_file:
        
        writer = csv.DictWriter(csv_file , fieldnames=results.keys())

        if not file_exists:
            writer.writeheader()

        writer.writerow(results)
        
    print(f"Results saved to {file_name}")